#SBI tests

In [ ]:
!pip install pytest torch transformers Pillow numpy scikit-learn pandas accelerate bitsandbytes -q

In [ ]:
# ===========================================================================
# 0. SBI Module Tests
# (per Test Plan in project book, Section 3.4.4)
# ===========================================================================

class TestSBIModule:

    # -----------------------------------------------------------------------
    # SBILoadImage
    # -----------------------------------------------------------------------
    def test_load_image(self, tmp_path):
        """
        SBILoadImage:
        Verify that a standard image loads from disk and converts to RGB
        without errors, and that the result is a valid PIL Image.
        """
        img_path = str(tmp_path / "face.jpg")
        Image.new('RGB', (256, 256), color=(120, 80, 60)).save(img_path)

        img = Image.open(img_path).convert('RGB')
        assert img is not None
        assert img.mode == 'RGB'
        assert img.size == (256, 256)

    # -----------------------------------------------------------------------
    # SBILandmarkDetection
    # -----------------------------------------------------------------------
    def test_landmark_detection(self):
        """
        SBILandmarkDetection:
        Verify that the landmark detection step produces an array of shape
        (N, 2) where each point is a valid (x, y) pixel coordinate within
        the image bounds.

        Uses a synthetic landmark array (as real face detection requires
        a face image + detector). This test validates the interface contract
        that the rest of the SBI pipeline depends on.
        """
        image_w, image_h = 224, 224
        num_landmarks    = 81

        # Simulate output of a face landmark detector
        synthetic_landmarks = np.random.randint(
            low=0, high=min(image_w, image_h), size=(num_landmarks, 2)
        ).astype(np.float32)

        assert synthetic_landmarks.shape == (num_landmarks, 2), \
            f"Landmarks must be shape ({num_landmarks}, 2)"

        assert synthetic_landmarks[:, 0].max() < image_w, \
            "x coordinates must be within image width"
        assert synthetic_landmarks[:, 1].max() < image_h, \
            "y coordinates must be within image height"
        assert synthetic_landmarks.min() >= 0, \
            "Landmark coordinates must be non-negative"

    # -----------------------------------------------------------------------
    # SBIMaskGeneration
    # -----------------------------------------------------------------------
    def test_mask_generation(self):
        """
        SBIMaskGeneration:
        Verify that the blending mask is a valid non-trivial grayscale array:
        - correct spatial shape
        - values in [0, 1]
        - not all-zero (empty mask would produce no blending)
        - not all-one  (full mask would be a trivial blend)

        Uses the convex hull blending logic from SBI (Shiohara & Yamasaki, 2022).
        """
        from scipy.spatial import ConvexHull
        import cv2

        image_size = 224

        # Simulate face landmarks roughly in the center of the image
        rng = np.random.default_rng(42)
        landmarks = rng.integers(60, 160, size=(20, 2)).astype(np.float32)

        # Build convex hull mask (core SBI step)
        hull = ConvexHull(landmarks)
        hull_points = landmarks[hull.vertices].astype(np.int32)
        mask = np.zeros((image_size, image_size), dtype=np.float32)
        cv2.fillConvexPoly(mask, hull_points, 1.0)

        assert mask.shape == (image_size, image_size), \
            "Mask must match image spatial dimensions"
        assert mask.min() >= 0.0 and mask.max() <= 1.0, \
            "Mask values must be in [0, 1]"
        assert mask.sum() > 0, \
            "Mask must not be all-zero (no blending region detected)"
        assert mask.mean() < 1.0, \
            "Mask must not be all-one (trivial blend)"

    # -----------------------------------------------------------------------
    # SBIBoundaryCheck
    # -----------------------------------------------------------------------
    def test_boundary_check(self):
        """
        SBIBoundaryCheck:
        Verify that the SBI blending operation:
        1. Preserves the correct output image size (224x224x3)
        2. Pixel values remain in valid uint8 range [0, 255]
        3. The blended image is not identical to either source or target
           (i.e., blending actually occurred)

        Formula from project book: I_SB = I_S ⊙ M + I_T ⊙ (1 - M)
        """
        H, W = 224, 224
        rng = np.random.default_rng(0)

        source = rng.integers(0, 255, (H, W, 3), dtype=np.uint8).astype(np.float32)
        target = rng.integers(0, 255, (H, W, 3), dtype=np.uint8).astype(np.float32)

        # Simple center-region mask
        mask = np.zeros((H, W, 1), dtype=np.float32)
        mask[60:160, 60:160, :] = 0.75

        blended = (source * mask + target * (1.0 - mask)).astype(np.uint8)

        assert blended.shape == (H, W, 3), \
            f"Blended image must be ({H}, {W}, 3), got {blended.shape}"
        assert blended.min() >= 0 and blended.max() <= 255, \
            "Blended pixel values must be in uint8 range [0, 255]"
        assert not np.array_equal(blended, source[:H, :W, :].astype(np.uint8)), \
            "Blended image must differ from source"
        assert not np.array_equal(blended, target[:H, :W, :].astype(np.uint8)), \
            "Blended image must differ from target"

    # -----------------------------------------------------------------------
    # SBI formula correctness
    # -----------------------------------------------------------------------
    def test_sbi_blend_formula(self):
        """
        Verify the SBI blending formula I_SB = I_S ⊙ M + I_T ⊙ (1-M)
        at known pixel values to confirm arithmetic correctness.
        """
        source  = np.array([[[200, 100, 50]]], dtype=np.float32)  # 1x1x3
        target  = np.array([[[0,   0,   0]]], dtype=np.float32)
        mask    = np.array([[[0.5]]])

        blended = source * mask + target * (1.0 - mask)

        expected = np.array([[[100., 50., 25.]]])
        np.testing.assert_allclose(blended, expected, rtol=1e-5,
            err_msg="SBI blend formula output does not match expected values")


In [ ]:
!pytest test_deepfake_detector.py -v